In [1]:


#@title The MIT License (MIT)
#
# Copyright (c) 2026 Eric dos Santos.
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in
# all copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN
# THE SOFTWARE.

# Fake News Classification

This project aims to develop a neural network for detecting fake news in Portuguese, using the dataset [Fake.br-Corpus](https://github.com/roneysco/Fake.br-Corpus). With this, we seek to create a system capable of identifying patterns and distinguishing fake news from real news, contributing to the fight against misinformation.

<table class="tfo-notebook-buttons" align="center">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/ericshantos/br_fake_news_detector/blob/main/br_fake_news_detector_model.ipynb
"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run on Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/ericshantos/br_fake_news_detector/blob/main/br_fake_news_detector_model.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View the code on GitHub</a>
  </td>
</table>

## Dataset loading

In [2]:
from pathlib import Path
import pandas as pd
import os

In [3]:
!git clone https://github.com/roneysco/Fake.br-Corpus
DATA_PATH = Path("./Fake.br-Corpus/full_texts")

Cloning into 'Fake.br-Corpus'...
remote: Enumerating objects: 28763, done.
remote: Total 28763 (delta 0), reused 0 (delta 0), pack-reused 28763 (from 1)
Receiving objects: 100% (28763/28763), 37.10 MiB | 10.83 MiB/s, done.
Resolving deltas: 100% (14129/14129), done.
Updating files: 100% (21602/21602), done.


In [4]:
# News Directory
fake_dir = DATA_PATH / "fake"
real_dir = DATA_PATH / "true"

### News content extraction:


In [5]:
import os
import pandas as pd

def load_news(news_dir: str, label: str) -> pd.DataFrame:
    # List to store news
    news = []

    # Cycle through all files in the specified directory
    for filename in os.listdir(news_dir):
        # Checks if the file has the .txt extension
        if filename.endswith(".txt"):
            # Gets the full path of the file
            file_path = os.path.join(news_dir, filename)

            # Open the file and read its contents
            with open(file_path, "r") as file:
                content = file.read()

                # Adds the content and label to the news list
                news.append({"text": content, "label": label})

    # Returns a pandas DataFrame containing the news
    return pd.DataFrame(news)

Result:

In [6]:
fake_news = load_news(fake_dir, 0)
real_news = load_news(real_dir, 1)

## Data preprocessing

In [7]:
from torch.utils.data import Dataset, DataLoader

### Concatenate the DataFrames

Group Dataframes to generate a single robust database.

In [8]:
data_news = pd.concat(
  [fake_news, real_news], ignore_index=True
).sample(frac=1, random_state=13)

Final base information:

In [9]:
data_news.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7200 entries, 3248 to 338
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    7200 non-null   object
 1   label   7200 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 168.8+ KB


### Data cleaning

## Training

In [10]:
from transformers import AutoTokenizer
import torch

### Splitting the dataset into training and testing

In [11]:
from sklearn.model_selection import train_test_split

# Splits data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
  data_news['text'],
  data_news['label'],
  test_size=0.2,
  random_state=13,
  stratify=data_news['label']
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

Training set size: (5760,)
Test set size: (1440,)


### Create tokenizer

In [12]:
from transformers import AutoTokenizer

In [13]:
tokenizer = AutoTokenizer.from_pretrained(
  "neuralmind/bert-base-portuguese-cased"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## Architectures

In [14]:
class NewsDataset(Dataset):
  def __init__(self, texts, label, tokenizer, max_length=256):
    self.texts = texts.tolist() if hasattr(texts, 'tolist') else texts
    self.label = label.tolist() if hasattr(label, 'tolist') else label
    self.tokenizer = tokenizer

    self.max_length = max_length

  def __getitem__(self, idx: int):
    encoding = self.tokenizer(
        self.texts[idx],
        padding='max_length',
        truncation=True,
        max_length=self.max_length,
        return_tensors="pt"
    )

    return {
        "input_ids": encoding["input_ids"].squeeze(0),
        "attention_mask": encoding["attention_mask"].squeeze(0),
        "label": torch.tensor(self.label[idx], dtype=torch.long)
    }

  def __len__(self) -> int:
    return len(self.texts)

In [15]:
dataset_train = NewsDataset(X_train, y_train, tokenizer)
dataset_test = NewsDataset(X_test, y_test, tokenizer)

In [16]:
loader_train = DataLoader(
  dataset_train,
  batch_size=8,
  shuffle=True,
  pin_memory=True,
  num_workers=2,
  prefetch_factor=2
)

loader_test = DataLoader(
  dataset_test,
  batch_size=8,
  pin_memory=True,
  num_workers=2
)

### Model architecture

In [17]:
import torch.nn as nn
from transformers import AutoModel

In [18]:
class BERTClassifier(nn.Module):

  def __init__(self, drop_rate: int = 0.2) -> None:
    super().__init__()

    self.bert = AutoModel.from_pretrained(
      "neuralmind/bert-base-portuguese-cased"
    )

    for param in self.bert.embeddings.parameters():
        param.requires_grad = False

    hidden = self.bert.config.hidden_size

    self.classifier = nn.Sequential(
        nn.Linear(hidden, 32),
        nn.GELU(),
        nn.Dropout(drop_rate),

        nn.Linear(32, 1)
    )

  def forward(self, input_ids, attention_mask):
    outputs = self.bert(
      input_ids=input_ids,
      attention_mask=attention_mask
    )

    pooler = outputs.last_hidden_state[:, 0, :]
    return self.classifier(pooler)

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [20]:
model = BERTClassifier().to(device)

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Model compilation**:

In [21]:
import torch.optim as optim

In [22]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=5e-5)

### Training the model

In [23]:
from tqdm import tqdm

In [24]:
epochs = 5

In [25]:
epochs = 5
accumulation_steps = 2

for epoch in range(epochs):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    progress_bar = tqdm(loader_train, desc=f"Epoch {epoch+1}")

    for batch_idx, batch in enumerate(progress_bar):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids, attention_mask).squeeze()
        loss = criterion(outputs, labels.float())

        loss = loss / accumulation_steps
        loss.backward()

        total_loss += loss.item() * accumulation_steps

        if (batch_idx + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        progress_bar.set_postfix({
            "batch_loss": f"{loss.item() * accumulation_steps:.4f}"
        })

        if batch_idx % 50 == 0:
            torch.cuda.empty_cache()

    if (batch_idx + 1) % accumulation_steps != 0:
        optimizer.step()
        optimizer.zero_grad()

    avg_loss = total_loss / len(loader_train)
    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")

Epoch 1:   0%|          | 0/720 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Epoch 1: 100%|██████████| 720/720 [04:17<00:00,  2.79it/s, batch_loss=0.0801]


Epoch 1 | Avg Loss: 0.1174


Epoch 2: 100%|██████████| 720/720 [04:20<00:00,  2.76it/s, batch_loss=0.0031]


...

Epoch 3 | Avg Loss: 0.0238


Epoch 4: 100%|██████████| 720/720 [04:20<00:00,  2.76it/s, batch_loss=0.0103]


Epoch 4 | Avg Loss: 0.0173


Epoch 5: 100%|██████████| 720/720 [04:20<00:00,  2.77it/s, batch_loss=0.0018]

Epoch 5 | Avg Loss: 0.0385


## Save to model

In [32]:
from transformers import AutoModelForSequenceClassification, AutoConfig

In [27]:
torch.save(model.state_dict(), "veritas-bert-ptbr.pth")

In [28]:
state_dict = torch.load("veritas-bert-ptbr.pth", map_location=device)

In [35]:
model = BERTClassifier(drop_rate=0.2)
model.load_state_dict(state_dict)
model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(29794, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_af

## Model evaluation

In [36]:
y_true = []
y_pred = []
y_prob = []

model.eval()

with torch.no_grad():
    for batch in loader_test:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids, attention_mask)

        probs = torch.sigmoid(logits)

        preds = (probs > 0.8).long()

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs.cpu().numpy())

In [37]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

In [39]:
print("Acurácia:", accuracy_score(y_true, y_pred))
print("Precisão:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1:", f1_score(y_true, y_pred))

print("\nRelatório completo:")
print(classification_report(y_true, y_pred))

print("\nMatriz de confusão:")
print(confusion_matrix(y_true, y_pred))

print("\nROC-AUC:")
print(roc_auc_score(y_true, y_prob))

Acurácia: 0.9861111111111112
Precisão: 0.9781420765027322
Recall: 0.9944444444444445
F1: 0.9862258953168044

Relatório completo:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99       720
           1       0.98      0.99      0.99       720

    accuracy                           0.99      1440
   macro avg       0.99      0.99      0.99      1440
weighted avg       0.99      0.99      0.99      1440


Matriz de confusão:
[[704  16]
 [  4 716]]

ROC-AUC:
0.9985580632716049
